# FER-2013 GNN — MLP Baseline (SGU 2026)

Facial Expression Recognition via Pixel Graph

---

## Truoc khi chay, can add dataset vao notebook

| Dataset name | Noi dung |
|---|---|
| `fer-graph-repo` | Graph repository (chunks) — build tu may local |

**Cach add:** Notebook -> `+ Add data` -> tim `fer-graph-repo` -> Add

**Luu y:** Sau khi add, Kaggle tu dong map dataset vao `/kaggle/input/`.
Path chinh xac phu thuoc vao ten dataset va username cua ban.
Notebook nay se **tu dong tim** path chua `shared_graph.pt`.

---

## Workflow

```
Local:
  python scripts/build_graph_repository.py ... -> artifacts/graph_repo/
  Upload artifacts/graph_repo/ len Kaggle dataset 'fer-graph-repo'

Kaggle (notebook nay):
  Cell 1: Kiem tra input, tu dong tim GRAPH_REPO_PATH
  Cell 2: Cai wandb, clone/pull repo code tu GitHub
  Cell 3: Kiem tra graph repository
  Cell 4: Inspect nhanh repository
  Cell 5: Train MLP Baseline
  Cell 6: Xem ket qua
```

In [ ]:
# =============================================================================
# Cell 1: Kiem tra input va tu dong tim GRAPH_REPO_PATH
# =============================================================================
import os

print("=" * 60)
print("Scanning /kaggle/input for shared_graph.pt ...")
print("=" * 60)

GRAPH_REPO_PATH = None

for dirname, dirs, files in os.walk("/kaggle/input"):
    if "shared_graph.pt" in files:
        # dirname la .../shared/, GRAPH_REPO_PATH la thu muc cha
        GRAPH_REPO_PATH = os.path.dirname(dirname)
        print(f"Found shared_graph.pt at: {dirname}")
        print(f"=> GRAPH_REPO_PATH = {GRAPH_REPO_PATH}")
        break

if GRAPH_REPO_PATH is None:
    print("[ERROR] Khong tim thay shared_graph.pt trong /kaggle/input!")
    print("Hay add dataset 'fer-graph-repo' vao notebook.")
    raise FileNotFoundError("shared_graph.pt not found under /kaggle/input")

print()
print("All input files:")
count = 0
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        fpath   = os.path.join(dirname, filename)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  [{size_mb:7.2f} MB]  {fpath}")
        count += 1
        if count >= 50:
            print("  ... (truncated at 50 files)")
            break
    if count >= 50:
        break

print(f"\nTotal files shown: {count}")

In [ ]:
# =============================================================================
# Cell 2: Cai wandb, lay secrets, clone/pull repo code tu GitHub
# =============================================================================
import os
from kaggle_secrets import UserSecretsClient

# --- Cai wandb ---
os.system("pip install wandb -q")

# --- Git config ---
os.system('git config --global user.email "phucga150625@gmail.com"')
os.system('git config --global user.name "Luphuc2005"')

# --- Constants ---
GITHUB_USERNAME    = "doduyquy"
GITHUB_REPO_NAME   = "sgu-2026-facial-expression-recognition"
GITHUB_REPO_BRANCH = "main"

# --- Lay secrets tu Kaggle Secrets ---
user_secrets  = UserSecretsClient()
GITHUB_TOKEN  = user_secrets.get_secret("GH_TOKEN")
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = WANDB_API_KEY

print("GH_TOKEN    :", "OK" if GITHUB_TOKEN  else "MISSING")
print("WANDB_API_KEY:", "OK" if WANDB_API_KEY else "MISSING")

# --- Clone hoac pull repo ---
REPO_URL  = "https://" + GITHUB_USERNAME + ":" + GITHUB_TOKEN + "@github.com/" + GITHUB_USERNAME + "/" + GITHUB_REPO_NAME + ".git"
WORK_DIR  = "/kaggle/working"
REPO_PATH = os.path.join(WORK_DIR, GITHUB_REPO_NAME)

os.chdir(WORK_DIR)

if not os.path.exists(REPO_PATH):
    print("--- Cloning repo ---")
    os.system("git clone -b " + GITHUB_REPO_BRANCH + " " + REPO_URL)
else:
    print("--- Repo exists, pulling latest ---")
    os.chdir(REPO_PATH)
    os.system("git pull origin " + GITHUB_REPO_BRANCH)

os.chdir("/kaggle/working/" + GITHUB_REPO_NAME)

import sys
REPO_DIR = os.getcwd()
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo ready:", os.getcwd())
print("sys.path[0]:", sys.path[0])

In [ ]:
# =============================================================================
# Cell 3: Kiem tra Graph Repository
# GRAPH_REPO_PATH duoc phat hien tu dong o Cell 1
# =============================================================================
import os
import torch

print("=" * 60)
print("Graph Repository Check")
print("Repo root:", GRAPH_REPO_PATH)
print("=" * 60)

all_ok = True

# 1. shared_graph.pt
shared_pt = os.path.join(GRAPH_REPO_PATH, "shared", "shared_graph.pt")
if os.path.exists(shared_pt):
    size_mb = os.path.getsize(shared_pt) / (1024 * 1024)
    shared  = torch.load(shared_pt, map_location="cpu", weights_only=False)
    print(f"  [OK] shared/shared_graph.pt ({size_mb:.2f} MB)")
    print(f"       grid={shared.height}x{shared.width}  conn={shared.connectivity}")
    print(f"       edges={shared.num_edges}  static_feats={shared.static_feature_names}")
else:
    print(f"  [MISSING] {shared_pt}")
    all_ok = False

# 2. tung split
for split in ["train", "val", "test"]:
    split_dir = os.path.join(GRAPH_REPO_PATH, split)
    if os.path.isdir(split_dir):
        chunks   = sorted([f for f in os.listdir(split_dir) if f.endswith(".pt")])
        total_mb = sum(os.path.getsize(os.path.join(split_dir, f)) for f in chunks) / (1024 * 1024)
        print(f"  [OK] {split:5s}/  {len(chunks):3d} chunks  ({total_mb:.1f} MB)")
    else:
        print(f"  [MISSING] {split_dir}")
        all_ok = False

# 3. manifest
manifest_pt = os.path.join(GRAPH_REPO_PATH, "manifest.pt")
if os.path.exists(manifest_pt):
    manifest = torch.load(manifest_pt, map_location="cpu", weights_only=False)
    print(f"  [OK] manifest.pt  version={manifest.get('version','?')}  built={manifest.get('built_at','?')}")
else:
    print("  [WARN] manifest.pt not found (non-critical)")

print()
if all_ok:
    print("Repository check PASSED")
else:
    raise RuntimeError("Repository check FAILED. See messages above.")

In [ ]:
# =============================================================================
# Cell 4: Inspect nhanh repository (tuy chon)
# Xem shapes, NaN, feature names cua sample dau tien
# =============================================================================
import subprocess

result = subprocess.run(
    [
        "python", "scripts/inspect_graph_repository.py",
        "--repo_root", GRAPH_REPO_PATH,
        "--split",     "train",
        "--chunk",     "0",
        "--sample",    "0",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# =============================================================================
# Cell 5: Train MLP Baseline
# Truyen GRAPH_REPO_PATH truc tiep qua --graph_repo_path
# (override gia tri mac dinh trong env.yaml, tranh loi sai path)
# =============================================================================
import os
import subprocess

print("Working dir :", os.getcwd())
print("Repo path   :", GRAPH_REPO_PATH)
print()

result = subprocess.run(
    [
        "python", "-m", "scripts.train",
        "--config",          "mlp_baseline",
        "--env",             "kaggle",
        "--dataloader_mode", "graph_vector",
        "--graph_repo_path", GRAPH_REPO_PATH,
    ],
    text=True,
)
print("Exit code:", result.returncode)

In [ ]:
# =============================================================================
# Cell 6: Xem ket qua dau ra (checkpoint, figures)
# =============================================================================
import os
import glob
from IPython.display import Image, display

OUTPUT_DIR = "/kaggle/working/sgu-2026-facial-expression-recognition/outputs"

# --- Checkpoints ---
print("=" * 55)
print("Checkpoint files")
print("=" * 55)
ckpt_root = os.path.join(OUTPUT_DIR, "checkpoints")
if os.path.isdir(ckpt_root):
    for root, dirs, files in os.walk(ckpt_root):
        for f in files:
            path    = os.path.join(root, f)
            size_mb = os.path.getsize(path) / (1024 * 1024)
            print(f"  {path}  ({size_mb:.2f} MB)")
else:
    print("  (no checkpoints yet)")

# --- Figures ---
print()
print("=" * 55)
print("Figure files")
print("=" * 55)
fig_root = os.path.join(OUTPUT_DIR, "figures")
if os.path.isdir(fig_root):
    figure_paths = sorted(glob.glob(os.path.join(fig_root, "**", "*.png"), recursive=True))
    if figure_paths:
        for img_path in figure_paths:
            rel_path = os.path.relpath(img_path, fig_root)
            print(f"  {rel_path}")
    else:
        print("  (no figures yet)")
else:
    print("  (no figures yet)")
    figure_paths = []

# --- Hien thi anh ---
for img_path in figure_paths:
    print()
    print("-" * 40)
    print(os.path.relpath(img_path, fig_root))
    print("-" * 40)
    display(Image(img_path))